# <div align="center"><b> CVAT Auto Etiquetado Manual </b></div>

<div align="right">

<!-- [![Binder](http://mybinder.org/badge.svg)](https://mybinder.org/) -->
[![nbviewer](https://img.shields.io/badge/render-nbviewer-orange?logo=Jupyter)](https://nbviewer.org)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com)

</div>

* * *

<style>
/* Limitar la altura de las celdas de salida en html */
.jp-OutputArea.jp-Cell-outputArea {
    max-height: 500px;
}
</style>

🛻 <em><font color='MediumSeaGreen'>  Instalaciones: </font></em> 🛻

Este notebook utiliza [Poetry](https://python-poetry.org/) para la gestión de dependencias.
Primero instala Poetry siguiendo las instrucciones de su [documentación oficial](https://python-poetry.org/docs/#installation).
Luego ejecuta el siguiente comando para instalar las dependencias necesarias y activar el entorno virtual:

- Bash:
```bash
poetry install
eval $(poetry env activate)
```

- PowerShell:
```powershell
poetry install
Invoke-Expression (poetry env activate)
```

<!-- Descargar archivos adicionales:
!gdown https://drive.google.com/drive/folders/1UBZ8PEbtmiWMGkULu7GAt3VhUpeTy9l7?usp=sharing --folder -->

In [1]:
# Chequear versión de CUDA
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Wed_Oct_30_01:18:48_Pacific_Daylight_Time_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


In [2]:
# Chequear más datos sobre la GPU
!nvidia-smi

Mon Oct 20 17:43:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.36                 Driver Version: 566.36         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
|  0%   38C    P8              4W /  320W |     506MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

✋ <em><font color='DodgerBlue'>Importaciones:</font></em> ✋

In [3]:
# Recarga automática de módulos en Jupyter Notebook
%reload_ext autoreload
%autoreload 2

In [5]:
# Dependencias del sistema
from pathlib import Path
import os, json, re

# Dependencias locales 
from modulo_ia.config import settings as CONFIG

# Dependencias propias
from modulo_utilidades.s3_comunication import procesador_s3
from modulo_utilidades.labeling import procesador_anotaciones_mongodb
from modulo_utilidades.utils.types import ImageMetadata

# Dependencias de terceros
from loguru import logger as LOGGER
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction
os.environ["FIFTYONE_PLUGINS_DIR"] = "E:\\Documentos\\Git Repositories\\uba-ceia-proy-final\\ceia-proyecto-final\\modulo-IA\\.venv\\Lib\\site-packages\\plugins\\operators"
import fiftyone as fo
fo.config.plugins_dir = "E:\\Documentos\\Git Repositories\\uba-ceia-proy-final\\ceia-proyecto-final\\modulo-IA\\.venv\\Lib\\site-packages\\plugins\\operators"

🔧 <em><font color='tomato'>Configuraciones:</font></em> 🔧


In [6]:
MODEL_FOLDER = CONFIG.folders.models_folder
MODEL_NAME = "coco_palm_dataset_v1.0_palm_detection_yolo11x_640_b7218a073a3942339689b2e7f4e0b543.pt"
# MODEL_NAME = "coco_palm_dataset_v1.0_rpw_detection_yolo11x_640_936350d45e85445a8fe0a325d54e9840.pt"
MODEL_PATH = MODEL_FOLDER / MODEL_NAME

DOWNLOAD_FOLDER = Path("downloads")
PREDICTIONS_FOLDER = DOWNLOAD_FOLDER / "predictions"
PREDICTIONS_FOLDER_DATASET = DOWNLOAD_FOLDER / "predictions_dataset"
OUTPUT_PATCHES_FOLDER = DOWNLOAD_FOLDER / "patches"

GROUP_ID = "GroupID2"  # ID del grupo de parches a descargar
PREFIX = f"patches/{GROUP_ID}"  # Prefijo a agregar: patches/GroupID2/

IMG_SIZE = 640  # Tamaño de la imagen

# NOTA: La clase 0 se toma como background en CVAT. No se debe usar.
CLASS_NAMES = {
    1: "palmera-sana",
}

CATEGORIES = [{"id": id, "name": name, "supercategory": ""} for id, name in CLASS_NAMES.items()]

CATEGORY_MAPPING = {1: 1} # 1 - palmera -> 0 - palmera-sana

# Configuraciones para SAHI
CONFIDENCE_THRESHOLD = 0.75
OVERLAP_HEIGHT_RATIO = 0.4
OVERLAP_WIDTH_RATIO = 0.4
POSTPROCESS_MATCH_THRESHOLD = 0.2
VISUAL_TEXT_SIZE = 2
VISUAL_RECTANGLE_THICKNESS = 2
POSTPROCESS_TYPE = "NMS"  # Opciones: "GREEDYNMM", "NMS", "NMM"

<div align="center">✨Datos del proyecto:✨</div>

<p></p>

<div align="center">

| **Subtitulo**   | Anotaciones masivas - Auto etiquetado CVAT - YoloV11                                                                                                        |
| --------------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| **Descrpción**  | <small>Notebook de detección de palmeras para realizar anotaciones masivas en CVAT</small>                                                                    |

</div>

# Chequeo de conexiones

In [ ]:
# Chequeo de conexión a MinIO
LOGGER.info("Chequeando conexión a MinIO...")
procesador_s3.test_connection()

# Chequeo de conexión a MongoDB
LOGGER.info("Chequeando conexión a MongoDB...")
procesador_anotaciones_mongodb.test_connection()  # Verifica la conexión a MongoDB
LOGGER.success("Conexión a MongoDB verificada correctamente.")

# Descarga del dataset

In [ ]:
patches_list = procesador_anotaciones_mongodb.list_patches_in_group(group_id=GROUP_ID)
patches_metadata = [ImageMetadata(image_name=patch_name, group_id=GROUP_ID) for patch_name in patches_list]

OUTPUT_PATCHES_FOLDER = procesador_s3.download_patches_from_s3(patches_metadata)

## Funciones auxiliares

In [ ]:
def predict_with_slicing(sample, label_field, detection_model, **kwargs):
    sliced_result = get_sliced_prediction(image=sample.filepath, detection_model=detection_model, verbose=0, **kwargs)
    detections = [det for det in sliced_result.to_fiftyone_detections() if det.confidence > CONFIDENCE_THRESHOLD]
    sample[label_field] = fo.Detections(detections=detections)

# Inferencia del dataset

- <small>https://docs.voxel51.com/tutorials/small_object_detection.html</small>

## Cargar el dataset

In [ ]:
dataset = fo.Dataset.from_images_dir(
    images_dir=OUTPUT_PATCHES_FOLDER,
    name="patches_dataset",
    overwrite=True
)

## Cargar el modelo de detección

In [ ]:
model = YOLO(MODEL_PATH)

# Cargamos el modelo en el AutoDetectionModel de SAHI
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(MODEL_PATH),
    confidence_threshold=CONFIDENCE_THRESHOLD,
    image_size=IMG_SIZE,
    device="cuda"
)

## Probar predicción en una imagen

In [ ]:
# Ubicación de la imagen
image_path = OUTPUT_PATCHES_FOLDER / "RGB_MVD_2024_J-29-C-1-M-5_patch_22.jpg"

# Realizamos la predicción con slicing sobre la imagen
sliced_result = get_sliced_prediction(
    image=str(image_path),
    detection_model=detection_model,
    slice_height=IMG_SIZE,
    slice_width=IMG_SIZE,
    overlap_height_ratio=OVERLAP_HEIGHT_RATIO,
    overlap_width_ratio=OVERLAP_WIDTH_RATIO,
    postprocess_match_threshold=POSTPROCESS_MATCH_THRESHOLD,
    postprocess_type=POSTPROCESS_TYPE

)

# Exportamos la imagen
sliced_result.export_visuals(export_dir=PREDICTIONS_FOLDER, text_size=VISUAL_TEXT_SIZE, rect_th=VISUAL_RECTANGLE_THICKNESS)

## Realizar predicciones en el dataset

In [ ]:
kwargs = {
    "overlap_height_ratio": OVERLAP_HEIGHT_RATIO,
    "overlap_width_ratio": OVERLAP_WIDTH_RATIO,
    "postprocess_match_threshold": POSTPROCESS_MATCH_THRESHOLD,
    "postprocess_type": POSTPROCESS_TYPE
}

for sample in dataset.iter_samples(progress=True, autosave=True):
    predict_with_slicing(sample, label_field="predictions", detection_model=detection_model, slice_height=IMG_SIZE, slice_width=IMG_SIZE, **kwargs)

## Exportar dataset

In [ ]:
dataset.export(
    export_dir=str(PREDICTIONS_FOLDER_DATASET),
    dataset_type=fo.types.COCODetectionDataset,
    label_field="predictions",
    overwrite=True
)

## Copiar el dataset a calidad (para visualización)

In [ ]:
session = fo.launch_app(dataset)
session.freeze()

## Modificar archivo labels.json

In [ ]:
# Leemos el archivo JSON
with open(PREDICTIONS_FOLDER_DATASET / "labels.json", "r") as json_file:
    data = json.load(json_file)

# Modificamos las categorías
data["categories"] = CATEGORIES

# Modificamos el nombre de las imágenes (se le agrega un prefijo)
for image in data["images"]:
    image_name = re.sub(r"_patch_.*", "", image["file_name"])
    file_name = f"{PREFIX}/{image_name}/{image['file_name']}"
    image["file_name"] = file_name

# Modificamos las anotaciones (cambiamos el category_id)
for annotation in data["annotations"]:
    if annotation["category_id"] in CATEGORY_MAPPING:
        annotation["category_id"] = CATEGORY_MAPPING[annotation["category_id"]]
    else:
        raise ValueError(f"El category_id {annotation['category_id']} no está en el mapeo.")

# Guardamos el archivo JSON modificado
with open(PREDICTIONS_FOLDER_DATASET / "annotations.json", "w") as json_file:
    json.dump(data, json_file, indent=4)

LOGGER.success(f"Categorías modificadas: {CATEGORIES}")
LOGGER.success(f"Prefijo agregado: {PREFIX}")
LOGGER.success(f"Archivo annotations.json modificado y guardado en {PREDICTIONS_FOLDER_DATASET / 'annotations.json'}")